<a href="https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can observed search performance signals be used to identify and prioritize content pages that are strong candidates for refresh or further review?

### Decision Supported

This analysis supports the decision of which content pages should be prioritized for review or refresh based on their observed search performance signals.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

This project uses the FlyRank ML Internship Search Intelligence warehouse dataset. The analysis focuses on observed Google Search Console performance signals used for content opportunity scoring.

The main signals used are:
- GSC impressions
- GSC clicks
- GSC average position

The analysis uses observed search performance data from the available March 2026 window.

Fields that could introduce future information or label leakage are excluded from the baseline and modeling features. In particular, future-window trend fields and label-derived fields are not used as predictive inputs.

No client names, domains, private queries, credentials, or raw private exports are included in the public-facing analysis.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

#### Objective

The objective is to rank content pages by their opportunity for refresh or further review using observed search performance signals.

#### Features

The model uses observed search performance features including:

- GSC impressions
- GSC clicks
- GSC average position

These signals are available from the observation window and do not depend on future outcomes.

#### Baseline

The Week 4 rule-based baseline ranks content using observed GSC impressions, clicks, and average position. The resulting baseline contains 176,738 ranked content items.

The machine-learning approach will be evaluated against this baseline using the same evaluation data.

#### Label

The modeling target represents whether a content item shows the selected opportunity outcome in the defined evaluation window. The target is constructed separately from the prediction features so that future information is not used as an input.

#### Validation Design

The model will be evaluated using a holdout validation design. Training and evaluation data will be separated before model evaluation, and the same evaluation population will be used when comparing the model with the baseline.

#### Leakage Checks

Future-derived fields and label-derived fields are excluded from the model features. In particular, fields such as:

- `is_declining_label`
- `trend_direction`
- `trend_pct`

are not used as predictive features.

The model therefore relies only on signals that would have been available at prediction time.

#### Assumptions

The resulting score should be interpreted as a prioritization signal rather than proof that refreshing a page will cause higher rankings, clicks, or traffic. The model supports review and decision-making; it does not establish causal impact.

In [7]:
from huggingface_hub import login

login()

In [3]:
from huggingface_hub import get_token

token = get_token()

print("Token found:", token is not None)


Token found: True


In [4]:
import os

os.environ["HF_TOKEN"] = token

print("HF_TOKEN configured:", os.environ.get("HF_TOKEN") is not None)

HF_TOKEN configured: True


In [5]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{token}'
    );
""")

print("DuckDB authentication configured.")

DuckDB authentication configured.


In [6]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {REL}
""").df()

columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [8]:
df = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {REL}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
""").df()

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


In [9]:
print("Rows:", len(df))
print("Unique content:", df["content_hash_id"].nunique())
print("Missing values:")
print(df.isna().sum())

Rows: 3611061
Unique content: 176738
Missing values:
report_date         0
client_hash_id      0
content_hash_id     0
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
dtype: int64


In [10]:
features = df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()

features.describe()

,gsc_impressions,gsc_clicks,gsc_avg_position
count,3.611061e+06,3.611061e+06,3.611061e+06
mean,7.772164e+01,2.275874e-01,1.582665e+01
std,2.498747e+02,1.277267e+00,1.985603e+01
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,0.000000e+00,3.742120e+00
50%,1.600000e+01,0.000000e+00,7.500000e+00
75%,6.200000e+01,0.000000e+00,2.020000e+01
max,4.008400e+04,2.740000e+02,4.980000e+02


In [11]:
date_check = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS unique_dates
    FROM {REL}
""").df()

date_check

,min_date,max_date,unique_dates
0,2026-03-01,2026-03-31,31


In [12]:
# Show the available tables in the FlyRank warehouse

tables = con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").df()

tables

,table_schema,table_name


In [13]:
print(con.sql("""
    SELECT *
    FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
    LIMIT 50
""").df())

                                                 file
0   hf://datasets/FlyRank/internship-warehouse/.gi...
1   hf://datasets/FlyRank/internship-warehouse/REA...
2   hf://datasets/FlyRank/internship-warehouse/dim...
3   hf://datasets/FlyRank/internship-warehouse/dim...
4   hf://datasets/FlyRank/internship-warehouse/fac...
5   hf://datasets/FlyRank/internship-warehouse/fac...
6   hf://datasets/FlyRank/internship-warehouse/fac...
7   hf://datasets/FlyRank/internship-warehouse/fac...
8   hf://datasets/FlyRank/internship-warehouse/fac...
9   hf://datasets/FlyRank/internship-warehouse/fac...
10  hf://datasets/FlyRank/internship-warehouse/fac...
11  hf://datasets/FlyRank/internship-warehouse/fac...
12  hf://datasets/FlyRank/internship-warehouse/fac...
13  hf://datasets/FlyRank/internship-warehouse/fac...
14  hf://datasets/FlyRank/internship-warehouse/fac...
15  hf://datasets/FlyRank/internship-warehouse/fac...
16  hf://datasets/FlyRank/internship-warehouse/fac...
17  hf://datasets/FlyRank/in

In [15]:
import pandas as pd

files = con.sql("""
    SELECT file
    FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
""").df()

pd.set_option("display.max_colwidth", None)

print(files.to_string(index=False))

                                                                                                  file
                                             hf://datasets/FlyRank/internship-warehouse/.gitattributes
                                                  hf://datasets/FlyRank/internship-warehouse/README.md
                                        hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
                                        hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance

In [16]:
APR_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

april_columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {APR_REL}
""").df()

april_columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [17]:
# Load March and April page-level GSC signals

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions AS march_impressions,
        gsc_clicks AS march_clicks,
        gsc_avg_position AS march_position
    FROM {REL}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
""").df()

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions AS april_impressions,
        gsc_clicks AS april_clicks,
        gsc_avg_position AS april_position
    FROM {APR_REL}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
""").df()

print("March rows:", len(march))
print("April rows:", len(april))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 3611061
April rows: 3901049


In [ ]:
comparison = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Pages present in both months:", len(comparison))
comparison.head()

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
